In [1]:
import pandas as pd
import os
import shutil
import json

Creating the directory structure to save Output for Images

In [2]:
output_data_path = os.path.join("output")
os.makedirs(output_data_path, exist_ok=True)
output_data_images_path = os.path.join("output", "images")
os.makedirs(output_data_images_path, exist_ok=True)

Reading the data

In [3]:
vqa_data = pd.read_csv('../data/path_open_vqa.csv')
vqa_data = vqa_data.rename(columns={vqa_data.columns[2]: 'Image1_Path',
                                    vqa_data.columns[26]: 'Image2_Path',
                                    vqa_data.columns[27]: 'Image3_Path',
                                    vqa_data.columns[28]: 'Image4_Path',
                                    vqa_data.columns[29]: 'Image5_Path',
                                    'Image 1 Magnification ': 'Image1_Mag',
                                    'Image 2 Magnification ': 'Image2_Mag',
                                    'Image 3 Magnification ': 'Image3_Mag',
                                    'Image 4 Magnification ': 'Image4_Mag',
                                    'Image 5 Magnification ': 'Image5_Mag'})

# Converting all the NaN values to None values to mark the missing values rather than being read as 'nan'
vqa_data = vqa_data.astype(object).where(pd.notnull(vqa_data), None)
vqa_data = vqa_data.reset_index()
vqa_data['index'] = vqa_data['index'] + 1
vqa_data = vqa_data.rename(columns={'index': 'Case_ID'})
vqa_data.head()

,Case_ID,Timestamp,Pathologist ID,Image1_Path,Organ,Categorization,Regional Anatomy,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Answer 2,...,Open Ended - Wrong Answer 1,Open Ended - Wrong Answer 2,Image2_Mag,Image3_Mag,Image4_Mag,Image5_Mag,Image2_Path,Image3_Path,Image4_Path,Image5_Path
0,1,2/25/2025 13:41:20,CK,https://drive.google.com/open?id=1tgv50Q9W4Bm_...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,What is the primary architectural pattern obse...,Nodular pattern,Germinal centers are absent,...,None,None,None,None,None,None,None,None,None,None
1,2,2/25/2025 13:46:25,CK,https://drive.google.com/open?id=1igYpj4RL0XKx...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,These would best be characterized as small lym...,...,None,None,None,None,None,None,None,None,None,None
2,3,2/25/2025 13:51:37,CK,https://drive.google.com/open?id=1DKNZJQJ17SkX...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,"In this mantle cell lymphoma, what is the best...",The nuclei of the abnormal lymphocytes are mos...,This should be considered a highly cellular ti...,...,None,None,None,None,None,None,None,None,None,None
3,4,2/26/2025 11:30:23,CK,https://drive.google.com/open?id=1jUe0z6wlZ0s9...,Gastrointestinal - Small Intenstine,Infection (Benign),Intestinal villous mucosa,"In a child with poor weight gain, what are the...",Features that argue against celiac disease in ...,Granulomas here are often the consequence of u...,...,None,None,None,None,None,None,None,None,None,None
4,5,2/26/2025 11:42:26,CK,https://drive.google.com/open?id=1VNr78I5w671g...,Gastrointestinal - Dudenum,Infection (Benign),Intestinal villous mucosa,What are the main histologic features of a non...,Non-necrotizing granulomas are characterized b...,None,...,None,None,None,None,None,None,None,None,None,None


#### Image Magnification Issue

It was reported that the Pathologist CK didn't include the magnification of the eyepience lens (10x) while reporting the resolution of the images where pathologist AC included all those. So, to make this data consistent, resolutions collected by CK are being multiplied with 10 to ensure that resolution reflects the objective magnification of the microscope multiplied by the eyepiece lens magnification

Image Magnifications Reported by AC

In [4]:
vqa_data[vqa_data['Pathologist ID'] == 'AC'][['Image1_Mag', 'Image2_Mag', 'Image3_Mag']].apply(pd.Series.value_counts)

,Image1_Mag,Image2_Mag,Image3_Mag
100x,23,NaN,NaN
200x,19,NaN,NaN
20x,2,NaN,NaN
400x,4,NaN,NaN
40x,4,NaN,NaN


Image Magnifications Reported by CK

In [5]:
vqa_data[vqa_data['Pathologist ID'] == 'CK'][['Image1_Mag', 'Image2_Mag', 'Image3_Mag']].apply(pd.Series.value_counts)

,Image1_Mag,Image2_Mag,Image3_Mag
10X,1.0,NaN,NaN
10x,31.0,6.0,10.0
20x,33.0,11.0,10.0
2x,13.0,7.0,2.0
40x,8.0,3.0,2.0
4X,NaN,NaN,1.0
4x,16.0,10.0,8.0
5X,1.0,NaN,NaN
5x,2.0,NaN,4.0


This shows that CK did not incorporate the magnification of eyepiece lens (10x) when reporting the overall image resolution. So multiplying all the magnifications reported by CK by 10 will make our data consistent

In [6]:
def convert_mag(mag):
    path_id = mag.iloc[0]
    image_mag = mag.iloc[1]
    if pd.isnull(image_mag):
        return image_mag
    
    image_mag = image_mag.lower().strip()
    if path_id == 'CK':    
        if 'x' in image_mag:
            return str(int(image_mag.split('x')[0]) * 10) + 'x'
    return image_mag

In [7]:
vqa_data['Image1_Mag'] = vqa_data[['Pathologist ID', 'Image1_Mag']].apply(convert_mag, axis=1)
vqa_data['Image2_Mag'] = vqa_data[['Pathologist ID', 'Image2_Mag']].apply(convert_mag, axis=1)
vqa_data['Image3_Mag'] = vqa_data[['Pathologist ID', 'Image3_Mag']].apply(convert_mag, axis=1)
vqa_data.head()

,Case_ID,Timestamp,Pathologist ID,Image1_Path,Organ,Categorization,Regional Anatomy,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Answer 2,...,Open Ended - Wrong Answer 1,Open Ended - Wrong Answer 2,Image2_Mag,Image3_Mag,Image4_Mag,Image5_Mag,Image2_Path,Image3_Path,Image4_Path,Image5_Path
0,1,2/25/2025 13:41:20,CK,https://drive.google.com/open?id=1tgv50Q9W4Bm_...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,What is the primary architectural pattern obse...,Nodular pattern,Germinal centers are absent,...,None,None,None,None,None,None,None,None,None,None
1,2,2/25/2025 13:46:25,CK,https://drive.google.com/open?id=1igYpj4RL0XKx...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,These would best be characterized as small lym...,...,None,None,None,None,None,None,None,None,None,None
2,3,2/25/2025 13:51:37,CK,https://drive.google.com/open?id=1DKNZJQJ17SkX...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,"In this mantle cell lymphoma, what is the best...",The nuclei of the abnormal lymphocytes are mos...,This should be considered a highly cellular ti...,...,None,None,None,None,None,None,None,None,None,None
3,4,2/26/2025 11:30:23,CK,https://drive.google.com/open?id=1jUe0z6wlZ0s9...,Gastrointestinal - Small Intenstine,Infection (Benign),Intestinal villous mucosa,"In a child with poor weight gain, what are the...",Features that argue against celiac disease in ...,Granulomas here are often the consequence of u...,...,None,None,None,None,None,None,None,None,None,None
4,5,2/26/2025 11:42:26,CK,https://drive.google.com/open?id=1VNr78I5w671g...,Gastrointestinal - Dudenum,Infection (Benign),Intestinal villous mucosa,What are the main histologic features of a non...,Non-necrotizing granulomas are characterized b...,None,...,None,None,None,None,None,None,None,None,None,None


### Generating the Google Sheet Containing the mapping between Google Drive Links and Image Names

This will help to map the image names downloaded from collected data with the Google Drive Links we have in our data

In [8]:
# function listFilesInNestedFolders(parentFolderId) {
#   // Get the initial parent folder
#   var parentfolder_id = '1VdFeRx06PrTTCbeMVQ0wQ8ML8QF8sSxM-QWlTcMx7TKQAjpWRuW5-EWJi-DtuBDPeXTbGYNm'
#   const parentFolder = DriveApp.getFolderById(parentfolder_id);
#   var folderlisting = 'File Names and Links - '+ parentfolder_id;

#   var ss = SpreadsheetApp.create(folderlisting);
#   var sheet = ss.getActiveSheet();
#   sheet.appendRow(['name','link']);

#   // Get subfolders in the current folder and process them recursively
#   var subfolders = parentFolder.getFolders();
#   while (subfolders.hasNext()) {
#     var subfolder = subfolders.next();
#     // Get files in the current folder
#     var contents = subfolder.getFiles();
#     var file;
#     var name;
#     var link;
#     while (contents.hasNext()) {
#       file = contents.next();
#       name = file.getName();
#       link = file.getUrl();
#       sheet.appendRow([name,link]);
#     }
#   }
# }

Now Download the file generated which contains mapping between google drive links and names.

In [9]:
data_dir = "../data_eval"
file_name = "file_names_and_links_PathOpen_Images.csv"
file_path = os.path.join(data_dir, file_name)
google_drive_name_links = pd.read_csv(file_path)
google_drive_name_links['google_drive_image_id'] = google_drive_name_links['link'].apply(lambda x: x.split('/')[-1])
google_drive_name_links.head()

,name,link,google_drive_image_id
0,ck_HSP_5x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/1SYbssi84BMfkC...,1SYbssi84BMfkCcUAHSYiDOl7JSgTVKe1
1,ck_BALL_4x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/1XrSu7uRm3s-al...,1XrSu7uRm3s-alWtSOdgieuS2f2L2bnDd
2,ck_pschwannoma_20x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/1dGZ8MAgdKvT5w...,1dGZ8MAgdKvT5wd81FU8lMeXaVPJYMj_s
3,ck_CMV gastritis_4x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/11UMXpBZjiQdo0...,11UMXpBZjiQdo0cnHVeR_-rdISv_aMlnO
4,ck_rhabdo_5x - Chandra Krishnan.jpeg,https://drive.google.com/file/d/174myggtY1Fj5A...,174myggtY1Fj5Av5gFJmOnC9XZohKcN-e


Creating dict between Google Drive Image and Name

In [10]:
imageid_name_dict = dict(zip(google_drive_name_links['google_drive_image_id'], google_drive_name_links['name']))
imageid_name_dict

{'1SYbssi84BMfkCcUAHSYiDOl7JSgTVKe1': 'ck_HSP_5x - Chandra Krishnan.jpeg',
 '1XrSu7uRm3s-alWtSOdgieuS2f2L2bnDd': 'ck_BALL_4x - Chandra Krishnan.jpeg',
 '1dGZ8MAgdKvT5wd81FU8lMeXaVPJYMj_s': 'ck_pschwannoma_20x - Chandra Krishnan.jpeg',
 '11UMXpBZjiQdo0cnHVeR_-rdISv_aMlnO': 'ck_CMV gastritis_4x - Chandra Krishnan.jpeg',
 '174myggtY1Fj5Av5gFJmOnC9XZohKcN-e': 'ck_rhabdo_5x - Chandra Krishnan.jpeg',
 '1DGvPSPr7z63T7ly9QrlqwvQMY_sx4at3': 'ck_dura chl_10x - Chandra Krishnan.jpg',
 '19KsfXwd-HtN_ZOtYq3qzPcdKC9GxMUnR': 'ck_arpckd_10x - Chandra Krishnan.jpg',
 '1m3sG-JnULD4PxMWvZ8_eCk7IUxvGDwjB': 'ck_candida esophagitis_10x - Chandra Krishnan.jpg',
 '1XNAKCVMmz6oBbIRN0DKNXK9EbzhVkiZ1': 'ck_corpus luteum_4x - Chandra Krishnan.jpg',
 '1mIsQXUpuUFe5e5q8nW5f7sXpXCj3N95g': 'ck_lch_4x - Chandra Krishnan.jpeg',
 '1v0CWoU4lmdth83vf6JoCu62J0-IsVF_x': 'ck_midlineg_20x - Chandra Krishnan.jpeg',
 '13LyGLMR7gjLf75nZHOzP_iS1simU2Dxd': 'ck_breast lactation_4x - Chandra Krishnan.jpeg',
 '1rJeENEFUzryNuJefmzY_TA

#### Assigning the Image IDs and Image Names to all the Five images as per the case

In [11]:
def assign_image_id_name(image_path):
    case_id = image_path.iloc[0]

    # Get the image path for the first image
    image1_path = image_path.iloc[1]
    image1_id = None
    image1_name = None
    if pd.notnull(image1_path):
        image1_id  = 'img_pathopen_' + str(case_id) + '_01'
        image1_drive_link = image1_path.split('=')[-1]
        image1_name = imageid_name_dict.get(image1_drive_link)

    # Get the image path for the second image
    image2_path = image_path.iloc[2]
    image2_id = None
    image2_name = None
    if pd.notnull(image2_path):
        image2_id  = 'img_pathopen_' + str(case_id) + '_02'
        image2_drive_link = image2_path.split('=')[-1]
        image2_name = imageid_name_dict.get(image2_drive_link)

    # Get the image path for the third image
    image3_path = image_path.iloc[3]
    image3_id = None
    image3_name = None
    if pd.notnull(image3_path):
        image3_id  = 'img_pathopen_' + str(case_id) + '_03'
        image3_drive_link = image3_path.split('=')[-1]
        image3_name = imageid_name_dict.get(image3_drive_link)

    # Get the image path for the fourth image
    image4_path = image_path.iloc[4]
    image4_id = None
    image4_name = None
    if pd.notnull(image4_path):
        image4_id  = 'img_pathopen_' + str(case_id) + '_04'
        image4_drive_link = image4_path.split('=')[-1]
        image4_name = imageid_name_dict.get(image4_drive_link)

    # Get the image path for the fifth image
    image5_path = image_path.iloc[5]
    image5_id = None
    image5_name = None
    if pd.notnull(image5_path):
        image5_id  = 'img_pathopen_' + str(case_id) + '_05'
        image5_drive_link = image5_path.split('=')[-1]
        image5_name = imageid_name_dict.get(image5_drive_link)
    
    return image1_id, image2_id, image3_id, image4_id, image5_id, image1_name, image2_name, image3_name, image4_name, image5_name


In [12]:
vqa_data[['Image1_ID', 'Image2_ID', 'Image3_ID', 'Image4_ID', 'Image5_ID', 'Image1_Name', 'Image2_Name', 'Image3_Name', 'Image4_Name', 'Image5_Name']] = vqa_data[['Case_ID','Image1_Path', 'Image2_Path', 'Image3_Path', 'Image4_Path', 'Image5_Path']].apply(assign_image_id_name, axis=1, result_type='expand')
vqa_data.head(10)

,Case_ID,Timestamp,Pathologist ID,Image1_Path,Organ,Categorization,Regional Anatomy,Open Ended - Question 1,Open Ended - Answer 1,Open Ended - Answer 2,...,Image1_ID,Image2_ID,Image3_ID,Image4_ID,Image5_ID,Image1_Name,Image2_Name,Image3_Name,Image4_Name,Image5_Name
0,1,2/25/2025 13:41:20,CK,https://drive.google.com/open?id=1tgv50Q9W4Bm_...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,What is the primary architectural pattern obse...,Nodular pattern,Germinal centers are absent,...,img_pathopen_1_01,None,None,None,None,MCL_20x - Chandra Krishnan.jpg,None,None,None,None
1,2,2/25/2025 13:46:25,CK,https://drive.google.com/open?id=1igYpj4RL0XKx...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,What is the predominant cell type seen here?,The main cell type observed here is a lymphocy...,These would best be characterized as small lym...,...,img_pathopen_2_01,None,None,None,None,MCL_200x - Chandra Krishnan.jpg,None,None,None,None
2,3,2/25/2025 13:51:37,CK,https://drive.google.com/open?id=1DKNZJQJ17SkX...,Hematolymphoid - Lymph Nodes,Neoplasia (Malignant),Lymph node,"In this mantle cell lymphoma, what is the best...",The nuclei of the abnormal lymphocytes are mos...,This should be considered a highly cellular ti...,...,img_pathopen_3_01,None,None,None,None,MCL_400x - Chandra Krishnan.jpg,None,None,None,None
3,4,2/26/2025 11:30:23,CK,https://drive.google.com/open?id=1jUe0z6wlZ0s9...,Gastrointestinal - Small Intenstine,Infection (Benign),Intestinal villous mucosa,"In a child with poor weight gain, what are the...",Features that argue against celiac disease in ...,Granulomas here are often the consequence of u...,...,img_pathopen_4_01,None,None,None,None,Duodenum granuloma_40x - Chandra Krishnan.jpg,None,None,None,None
4,5,2/26/2025 11:42:26,CK,https://drive.google.com/open?id=1VNr78I5w671g...,Gastrointestinal - Dudenum,Infection (Benign),Intestinal villous mucosa,What are the main histologic features of a non...,Non-necrotizing granulomas are characterized b...,None,...,img_pathopen_5_01,None,None,None,None,Duodenum granuloma_100x - Chandra Krishnan.jpg,None,None,None,None
5,6,2/26/2025 11:46:09,CK,https://drive.google.com/open?id=1l5dsr6VVkG4P...,Gastrointestinal - Dudenum,Normal,Duodenum,"In villous tissue, what is the expected mixtur...",In the lamina propria of normal villous tissue...,None,...,img_pathopen_6_01,None,None,None,None,Duodenum granuloma_200x - Chandra Krishnan.jpg,None,None,None,None
6,7,2/27/2025 15:22:33,CK,https://drive.google.com/open?id=1KDuoZjxg4KTl...,Skin,Infection (Benign),Squamous epithelium,What does the presence of increased compact la...,Excess anucleate keratin on the skin surface i...,"Those refractile structures are hair shafts, w...",...,img_pathopen_7_01,None,None,None,None,wart_40x - Chandra Krishnan.jpg,None,None,None,None
7,8,2/27/2025 15:29:29,CK,https://drive.google.com/open?id=1BW4uJGlQx_0r...,Skin,Infection (Benign),Squamous epithelium,How does increased intracellular fluid manifes...,Spongiosis is the alteration of squamous epith...,Basal keratinocytes are epithelioid in shape w...,...,img_pathopen_8_01,None,None,None,None,wart_200x - Chandra Krishnan.jpg,None,None,None,None
8,9,3/6/2025 10:56:20,CK,https://drive.google.com/open?id=1GQHppS17EQxo...,Head/Neck - congenital,Congenital abnormality,Subcutaneous tissues,What is the likely clinical scenario that this...,The findings here would represent a branchial ...,The typical origin for this histology is from ...,...,img_pathopen_9_01,None,None,None,None,ck_branchial arch_20x - Chandra Krishnan.jpg,None,None,None,None
9,10,3/10/2025 9:57:15,CK,https://drive.google.com/open?id=14rsOnN9AySFK...,Pediatric - congenital,Hamartoma,Hyaline cartilage,The image shows a mixture of epithelial and me...,These are features typical of a branchial arch...,The green coloration represents the applicatio...,...,img_pathopen_10_01,None,None,None,None,ck_branchial arch_40x - Chandra Krishnan.jpg,None,None,None,None


Validating our operations

In [13]:
assert len(vqa_data[vqa_data['Image1_Path'].notnull() & vqa_data['Image1_ID'].notnull() & vqa_data['Image1_Name'].notnull()]) == 157
assert len(vqa_data[vqa_data['Image2_Path'].notnull() & vqa_data['Image2_ID'].notnull() & vqa_data['Image2_Name'].notnull()]) == 37
assert len(vqa_data[vqa_data['Image2_Path'].notnull() & vqa_data['Image2_ID'].notnull() & vqa_data['Image2_Name'].notnull()]) == 37

### Preparing the dataset in Json Format

Specifying the directories of original images and augmented images

In [14]:
original_dataset_dir = "../PathOPEN_Images"
augmented_dataset_dir = "../PathOPEN_Images_Augmented"

Putting all the values in a dictionary

In [15]:
pathopen_data_list = []

for index, row in vqa_data.iterrows():
    pathopen_dict = {}
    # Getting Case ID
    pathopen_dict["Case ID"] = int(row['Case_ID']) if pd.notnull(row['Case_ID']) else ''
    # Getting Organ
    pathopen_dict["Organ"] = row['Organ'] if pd.notnull(row['Organ']) else ''
    # Getting Categorization
    pathopen_dict["Categorization"] = row['Categorization'] if pd.notnull(row['Categorization']) else ''
    # Getting Regional Anatomy
    pathopen_dict["Regional Anatomy"] = row['Regional Anatomy']  if pd.notnull(row['Regional Anatomy']) else ''
    # Getting Open-Ended Question/Answer
    open_ended_quest1 = dict(
        {
            "Question" : row['Open Ended - Question 1'] if pd.notnull(row['Open Ended - Question 1']) else '',
            "Correct Answer" : row['Open Ended - Answer 1'] if pd.notnull(row['Open Ended - Answer 1']) else '',
            "Wrong Answer" : row['Open Ended - Wrong Answer 1'] if pd.notnull(row['Open Ended - Wrong Answer 1']) else '',
        })
    open_ended_quest2 = dict(
        {
            "Question" : row['Open Ended - Question 2'] if pd.notnull(row['Open Ended - Question 2']) else '',
            "Correct Answer" : row['Open Ended - Answer 2'] if pd.notnull(row['Open Ended - Answer 2']) else '',
            "Wrong Answer" : row['Open Ended - Wrong Answer 2'] if pd.notnull(row['Open Ended - Wrong Answer 2']) else '',
        })

    pathopen_dict["Open Ended"] = [open_ended_quest1, open_ended_quest2]

    # Getting MCQ
    pathopen_dict["MCQ"] = dict({
                                    "Question": row['MCQ - Question'] if pd.notnull(row['MCQ - Question']) else '',
                                    "Option 1": row['MCQ - Option 1'] if pd.notnull(row['MCQ - Option 1']) else '',            
                                    "Option 2": row['MCQ - Option 2'] if pd.notnull(row['MCQ - Option 2']) else '',            
                                    "Option 3": row['MCQ - Option 3'] if pd.notnull(row['MCQ - Option 3']) else '',            
                                    "Option 4": row['MCQ - Option 4'] if pd.notnull(row['MCQ - Option 4']) else '',            
                                    "Option 5": row['MCQ - Option 5'] if pd.notnull(row['MCQ - Option 5']) else '',
                                    "Answer": row['MCQ - Answer'] if pd.notnull(row['MCQ - Answer']) else ''
                                })

    # Getting Close-Ended
    pathopen_dict["Close Ended"] = dict({
                                            "Question": row['Close-Ended Question 1'] if pd.notnull(row['Close-Ended Question 1']) else '',
                                            "Answer": row['Close-Ended Answer 1'] if pd.notnull(row['Close-Ended Answer 1']) else ''
                                        })

    # Getting Images
    pathopen_images = []
    for i in range (1,6):
        image_id = row[f'Image{i}_ID']
        image_dir = ''
        image_mag = ''
        image_original_name = ''
        image_augmented_names = []

        if pd.notnull(image_id):
            image_out_path = os.path.join(output_data_images_path, image_id)
            image_name = row[f'Image{i}_Name']
            if pd.notnull(image_name):
                image_original_path = os.path.join(original_dataset_dir, image_name)
                image_augmented_dir_path = os.path.join(augmented_dataset_dir, f'{image_id}_augmented')
                if os.path.exists(image_original_path) and os.path.exists(image_augmented_dir_path):
                    # Create a directory to put the original and augmented images
                    os.makedirs(image_out_path, exist_ok=True)
                    image_dir = image_out_path
                    image_mag = row[f'Image{i}_Mag']
                    # Copy the original file with the changed name
                    image_original_name = f'{image_id}_orig.{image_name.split(".")[-1]}'
                    shutil.copy(image_original_path, os.path.join(image_out_path, image_original_name))
                    # Copy the augmented files into this folder
                    for file_name in os.listdir(image_augmented_dir_path):
                        src_file_path = os.path.join(image_augmented_dir_path, file_name)
                        dst_file_path = os.path.join(image_out_path, file_name)
                        image_augmented_names.append(file_name)
                        shutil.copy(src_file_path, dst_file_path)
        
            images_dict = dict({
                "Directory": image_dir,
                "Magnification": image_mag,
                "Original": image_original_name,
                "Augmented": image_augmented_names
            })
        
            pathopen_images.append(images_dict)

    pathopen_dict["Images"] = pathopen_images
    
    pathopen_data_list.append(pathopen_dict)

Saving the data in a json file

In [16]:
# Writing data to data.json
with open(os.path.join(output_data_path, "pathopen_data.json"), "w") as file:
    json.dump(pathopen_data_list, file, indent=4)